<a href="https://colab.research.google.com/github/Soheilp86/Statistical-Machine-Learning/blob/main/61_Neural_Networks_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks with scikit-learn

In this notebook we will

1. Decide when a neural network is better than logistic regression,
2. Train a neural network on handwritten digits,
3. Look inside the network and learn its main parameters,
4. **Your turn:** apply it to forest cover type data.

In [ ]:
## Data manipulation and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Datasets
from sklearn.datasets import make_blobs, make_moons, make_circles, load_digits, fetch_covtype

## Model creation and training
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

## Performance metrics
from sklearn.metrics import accuracy_score, confusion_matrix

## 1. **When Do We Need a Neural Network?**

Logistic regression separates two classes with a **straight line**.
A neural network with a hidden layer can draw a **curved** boundary.

### 1.1 Look at the data

Here are three data sets. Each point has two features, $x_1$ and $x_2$. The color is the class.

In [ ]:
# Three 2D data sets
X_lin, y_lin = make_blobs(n_samples=400, centers=2, cluster_std=1.5, random_state=3)
X_moon, y_moon = make_moons(n_samples=400, noise=0.2, random_state=0)
X_circ, y_circ = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=0)

datasets = {
    "A": (X_lin, y_lin),
    "B": (X_moon, y_moon),
    "C": (X_circ, y_circ),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, (X2, y2)) in zip(axes, datasets.items()):
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(f"Data set {name}")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
plt.show()

**Question:** For each data set, is logistic regression (one neuron) enough, or do we need a neural network? Why?

| Data set | Your guess | Why? |
|---|---|---|
| A | | |
| B | | |
| C | | |

### 1.2 Check your guesses

We train both models on each data set and compare their test accuracy.

In [ ]:
def plot_boundary(model, X2, y2, ax, title):
    # color every point of a grid by the model's prediction
    xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 300),
                         np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="coolwarm", edgecolor="k", s=20)
    ax.set_title(title)

In [ ]:
results = []
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, (name, (X2, y2)) in enumerate(datasets.items()):
    X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=0)

    lr2 = LogisticRegression().fit(X2_train, y2_train)
    nn2 = MLPClassifier(hidden_layer_sizes=(32,), max_iter=2000, random_state=0).fit(X2_train, y2_train)

    lr_acc = lr2.score(X2_test, y2_test)
    nn_acc = nn2.score(X2_test, y2_test)
    results.append({"data set": name, "logistic regression": round(lr_acc, 3), "neural network": round(nn_acc, 3)})

    plot_boundary(lr2, X2, y2, axes[0, col], f"{name}: logistic regression ({lr_acc:.2f})")
    plot_boundary(nn2, X2, y2, axes[1, col], f"{name}: neural network ({nn_acc:.2f})")

plt.tight_layout()
plt.show()
pd.DataFrame(results)

- **Straight boundary** (data set A): logistic regression is enough.
- **Curved boundary** (data sets B and C): the neural network does better, because its hidden layer can bend the boundary.

With 2 features we can **see** the data. With many features we cannot, so we train both models and compare.

## 2. **Digits: The Data**

The digits data set has 1,797 images of handwritten digits (0 to 9).
Each image is an 8 × 8 grid of pixels. Each pixel is a number from 0 (white) to 16 (black).

In [ ]:
digits = load_digits()

# Show the first 10 images
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.show()

**Images as vectors.** Logistic regression and neural networks need each data point as a **vector**.
So each 8 × 8 image must become a vector of 8 × 8 = **64** numbers.
Usually we do this ourselves. Here, scikit-learn has already done it: `digits.data` holds the vectors.

In [ ]:
X = digits.data      # one row per image: 64 numbers
y = digits.target    # the digit in each image

print("X shape:", X.shape)
print()
print("First 3 rows of X:")
print(X[:3])

- `X` is a table of vectors: **1,797 rows**, one per image.
- Each row has **64 numbers**, one per pixel.
- So the network has **64 inputs** and **10 outputs** (one per digit).

## 3. **Digits: Logistic Regression vs. Neural Network**

First we split the data and scale the features. Neural networks train better when every feature has mean 0 and standard deviation 1.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_test_images = X_test.copy()   # unscaled copy, for plotting later

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn mean and std from the training data
X_test = scaler.transform(X_test)         # apply the same scaling to the test data

Now we train both models.

`MLPClassifier` is scikit-learn's neural network. `hidden_layer_sizes=(32,)` means one hidden layer with 32 neurons.

In [ ]:
# Logistic regression
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train, y_train)

# Neural network
mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1000, random_state=0)
mlp.fit(X_train, y_train)

print(f"Logistic regression: train {lr.score(X_train, y_train):.3f}, test {lr.score(X_test, y_test):.3f}")
print(f"Neural network:      train {mlp.score(X_train, y_train):.3f}, test {mlp.score(X_test, y_test):.3f}")

**Discussion:** Logistic regression does about as well as the neural network on the digits. Why? Use what you learned in Section 1.

## 4. **Digits: Using the Network**

### 4.1 Predictions

In [ ]:
y_pred = mlp.predict(X_test)

plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix: Neural Network on Digits")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

The images the network got wrong:

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print("Mistakes:", len(wrong), "out of", len(y_test))

fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(14, 2.2))
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.imshow(X_test_images[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"True {y_test[i]}, pred {y_pred[i]}", fontsize=9)
    ax.axis("off")
plt.show()

Like logistic regression, the network gives a probability for each class. `predict` picks the largest.

In [ ]:
probs = mlp.predict_proba(X_test[:1])[0]
print("Probabilities:", np.round(probs, 3))
print("Predicted:", probs.argmax(), "  True:", y_test[0])

### 4.2 Inside the network

- `mlp.coefs_`: one weight matrix per layer.
- `mlp.intercepts_`: one bias vector per layer.

Note: scikit-learn stores each weight matrix as (inputs × neurons), the **transpose** of the $W$ in the lecture.

In [ ]:
for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_), start=1):
    print(f"Layer {i}: weights {W.shape}, biases {b.shape}")

Number of parameters (weights + biases):

- Hidden layer: 64 × 32 + 32 = 2,080
- Output layer: 32 × 10 + 10 = 330
- Total: **2,410**

In [ ]:
def count_params(model):
    # all weights plus all biases
    return sum(W.size + b.size for W, b in zip(model.coefs_, model.intercepts_))

print("Neural network parameters:", count_params(mlp))
print("Logistic regression parameters:", lr.coef_.size + lr.intercept_.size)

### 4.3 The loss curve

Training starts from random weights. Each **iteration** changes the weights a little to lower the **loss** (the error on the training data).

In [ ]:
plt.plot(mlp.loss_curve_)
plt.xlabel("Iteration")
plt.ylabel("Training loss")
plt.title("Loss curve")
plt.show()

### 4.4 Main parameters

| Parameter | What it controls | Default |
|---|---|---|
| `hidden_layer_sizes` | number of hidden layers and neurons in each | `(100,)` |
| `activation` | activation in the hidden layers: `"relu"`, `"tanh"`, `"logistic"`, `"identity"` | `"relu"` |
| `max_iter` | maximum number of training iterations | `200` |
| `learning_rate_init` | step size of each weight update | `0.001` |
| `alpha` | penalty on large weights (regularization) | `0.0001` |
| `early_stopping` | stop when the validation score stops improving | `False` |
| `random_state` | the random starting weights | `None` |

### 4.5 Tuning with cross-validation

`GridSearchCV` tries every combination of parameters and keeps the best one.

We put the scaler and the network in a `Pipeline`, so scaling is redone inside each fold. `mlp__alpha` means "the `alpha` of the step named `mlp`".

In [ ]:
# Unscaled data: the pipeline does the scaling
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(max_iter=1000, random_state=0)),
])

param_grid = {
    "mlp__hidden_layer_sizes": [(16,), (64,), (64, 32)],
    "mlp__alpha": [0.0001, 0.01, 1],
}

grid = GridSearchCV(pipe, param_grid, cv=3, n_jobs=-1)
grid.fit(X_train_raw, y_train)

print("Best parameters:", grid.best_params_)
print(f"Best cross-validation accuracy: {grid.best_score_:.3f}")
print(f"Test accuracy: {grid.score(X_test_raw, y_test):.3f}")

## 5. **Your Turn: Forest Cover Type**

Each row is a 30 m × 30 m patch of forest in Colorado. The goal is to predict which tree type covers it.

- **54 features:** 10 measurements (elevation, slope, distance to water, ...) and 44 yes/no columns (wilderness area, soil type).
- **7 classes:** 1 Spruce/Fir, 2 Lodgepole Pine, 3 Ponderosa Pine, 4 Cottonwood/Willow, 5 Aspen, 6 Douglas-fir, 7 Krummholz.

### 5.1 Take a sample

The full data set has 581,012 rows. Training on all of it takes too long for class, so we use a **random sample of 20,000 rows**.

- `fetch_covtype()` downloads the data (only the first time).
- `train_test_split(..., train_size=20000)` keeps 20,000 random rows. We ignore the rest.
- `stratify=y_all` keeps the same share of each tree type as the full data, so rare tree types still appear.

In [ ]:
forest = fetch_covtype()
X_all, y_all = forest.data, forest.target

# Keep 20,000 random rows with the same mix of tree types
X_f, _, y_f, _ = train_test_split(X_all, y_all, train_size=20000, stratify=y_all, random_state=0)

print("Sample shape:", X_f.shape)
print("Rows per tree type (1 to 7):", np.bincount(y_f)[1:])

### 5.2 Split and scale

Split the sample with `test_size=0.2`, `stratify=y_f`, `random_state=42`. Then scale with `StandardScaler`.

In [ ]:
# Split


# Scale

### 5.3 Logistic regression vs. neural network

1. Train `LogisticRegression(max_iter=2000)`.
2. Train `MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=0)`.
3. Print the training and test accuracy of both.

A `ConvergenceWarning` means training stopped before the loss settled. You can ignore it here.

In [ ]:
# Logistic regression


# Neural network


# Training and test accuracy

**Question:** Is the boundary between tree types closer to straight or curved? How do you know? Compare with the digits.

### 5.4 Confusion matrix

Plot the confusion matrix of the neural network. Which tree types does it confuse most often?

In [ ]:
# Confusion matrix

### 5.5 Learning rate and the loss curve

Train three networks with `hidden_layer_sizes=(100,)`, `max_iter=300` and `learning_rate_init` = `0.0001`, `0.001`, `0.01`.
Plot the three loss curves on one figure: call `plt.plot(model.loss_curve_, label=...)` for each, then `plt.legend()`. What do you notice?

In [ ]:
# Loss curves for three learning rates

### 5.6 Tune with cross-validation

Build a `Pipeline` with a `StandardScaler` and an `MLPClassifier(max_iter=300, random_state=0)`. Use `GridSearchCV` with `cv=3` to search over:

- `hidden_layer_sizes`: `(32,)`, `(128,)`, `(128, 64)`
- `alpha`: `0.0001`, `0.01`

Use the **unscaled** sample (`X_f`, `y_f`) split as in 5.2. Report the best parameters, the best cross-validation accuracy and the test accuracy.

This can take a few minutes.

In [ ]:
# Unscaled split


# Pipeline and parameter grid


# Grid search and results

### 5.7 Which model would you choose?

Compare logistic regression and your best neural network on this data. Which would you choose, and why? How is this different from the digits?